In [17]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline
import numpy
import torch.nn.functional as F
import pandas as pd

# Load pre-trained BERT model fine-tuned on NER
MODEL_NAME = "dbmdz/bert-large-cased-finetuned-conll03-english"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)


Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [14]:
# Input text
text = "Apple Inc. was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in Cupertino, California in 1976."
inputs = tokenizer(text, return_tensors="pt", truncation=True, is_split_into_words=False)

# Run model
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
probs = F.softmax(logits, dim=2)
predictions = torch.argmax(probs, dim=2)[0].tolist()
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
labels = model.config.id2label

# Merge subwords and collect named entities
entities = []
current_entity = ""
current_label = ""

for token, pred_id in zip(tokens, predictions):
    label = labels[pred_id]
    
    # Skip special tokens like [CLS] and [SEP]
    if token in tokenizer.all_special_tokens:
        continue

    # Check if token is a subword
    if token.startswith("##"):
        current_entity += token[2:]
    else:
        if current_entity and current_label != "O":
            entities.append((current_entity, current_label))
        current_entity = token
        current_label = label

# Add last entity if exists
if current_entity and current_label != "O":
    entities.append((current_entity, current_label))

# Print results
for entity, label in entities:
    print(f"{entity} -> {label}")

Apple -> I-ORG
Inc -> I-ORG
Steve -> I-PER
Jobs -> I-PER
Steve -> I-PER
Wozniak -> I-PER
Ronald -> I-PER
Wayne -> I-PER
Cupertino -> I-LOC
California -> I-LOC


In [18]:
df = pd.read_csv("data/Financial.csv")

In [19]:
labels = model.config.id2label

In [20]:

def extract_entities(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512, is_split_into_words=False)
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    probs = F.softmax(logits, dim=2)
    predictions = torch.argmax(probs, dim=2)[0].tolist()
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    entities = []
    current_entity = ""
    current_label = ""

    for token, pred_id in zip(tokens, predictions):
        label = labels[pred_id]
        if token in tokenizer.all_special_tokens:
            continue
        if token.startswith("##"):
            current_entity += token[2:]
        else:
            if current_entity and current_label != "O":
                entities.append((current_entity, current_label))
            current_entity = token
            current_label = label
    if current_entity and current_label != "O":
        entities.append((current_entity, current_label))

    return entities




In [23]:
# Process only first two rows
df_sample = df.head(2).copy()

In [24]:


# Apply NER on the Content column for just 2 rows
df_sample['Entities'] = df_sample['Content'].apply(lambda x: extract_entities(str(x)))



# Preview result
print(df_sample[['Title', 'Entities']])

                                   Title  \
0  TSX Slightly Down, Books Weekly Gains   
1          UnitedHealth Hits 4-week High   

                                            Entities  
0  [(TSX, I-MISC), (States, I-MISC), (Stock, I-OR...  
1          [(UnitedHealth, I-ORG), (States, I-MISC)]  


In [25]:
df_sample[['Title', 'Entities']]


,Title,Entities
0,"TSX Slightly Down, Books Weekly Gains","[(TSX, I-MISC), (States, I-MISC), (Stock, I-OR..."
1,UnitedHealth Hits 4-week High,"[(UnitedHealth, I-ORG), (States, I-MISC)]"


In [26]:
# Save sample output (optional)
df_sample.to_csv("ner_sample_output_bert.csv", index=False)

In [27]:
# Show all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [29]:
# Show full content in each column
pd.set_option('display.max_colwidth', None)

In [31]:
# Prevent wrapping of wide columns
pd.set_option('display.expand_frame_repr', False)

In [32]:
df_sample[['Title', 'Entities']]

,Title,Entities
0,"TSX Slightly Down, Books Weekly Gains","[(TSX, I-MISC), (States, I-MISC), (Stock, I-ORG), (MarketThe, I-ORG), (S, I-ORG), (&, I-ORG), (P, I-ORG), (TSX, I-MISC), (Nasdaq, I-MISC), (US, I-LOC), (TELUS, I-ORG), (International, I-ORG), (Pine, I-ORG), (Cliff, I-ORG), (Energy, I-ORG), (Stifel, I-ORG), (First, I-ORG), (Quantum, I-ORG), (Minerals, I-ORG), (Raymond, I-ORG), (James, I-ORG)]"
1,UnitedHealth Hits 4-week High,"[(UnitedHealth, I-ORG), (States, I-MISC)]"
